In [19]:
import pandas as pd
import numpy as np
import re
from ast import literal_eval
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from defenders.pii_detection.hmm.HMM import HMM



In [20]:
NER_TAGS = [
    'B-ACCOUNTNAME', 'B-ACCOUNTNUMBER', 'B-CREDITCARDNUMBER', 'B-EMAIL',
    'B-IP', 'B-IPV4', 'B-IPV6', 'B-MAC', 'B-PASSWORD', 'B-PHONE_NUMBER',
    'B-SSN', 'B-USERNAME',
    'I-ACCOUNTNAME', 'I-ACCOUNTNUMBER', 'I-CREDITCARDNUMBER', 'I-EMAIL',
    'I-IP', 'I-IPV4', 'I-IPV6', 'I-MAC', 'I-PASSWORD', 'I-PHONE_NUMBER',
    'I-SSN', 'I-USERNAME',
    'O',
]

NER_TAG_TO_INDEX = {tag: i for i, tag in enumerate(NER_TAGS)}
NER_TAG_TO_INDEX['UNK'] = len(NER_TAGS)
INDEX_TO_NER_TAG = {i: tag for tag, i in NER_TAG_TO_INDEX.items()}


In [21]:
df = pd.read_parquet('../data/word_based_data.parquet')

train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.1, random_state=42)
print(len(train_df), len(val_df), len(test_df))


17486 1943 2159


In [22]:
def encode_tags(tag_seqs):
    return [[NER_TAG_TO_INDEX.get(t, NER_TAG_TO_INDEX['O']) for t in seq] for seq in tag_seqs]

def encode_words_baseline(word_seqs, vocab):
    return [[vocab.get(w, vocab['UNK']) for w in seq] for seq in word_seqs]


In [23]:

train_words_raw = [list(x) for x in train_df['words'].tolist()]
train_tags_raw  = [list(x) for x in train_df['labels'].tolist()]




In [24]:

baseline_vocab = {}
for seq in train_words_raw:
    for w in seq:
        if w not in baseline_vocab:
            baseline_vocab[w] = len(baseline_vocab)
baseline_vocab['UNK'] = len(baseline_vocab)

In [25]:

train_tags  = encode_tags(train_tags_raw)
train_obs   = encode_words_baseline(train_words_raw, baseline_vocab)


In [26]:

test_words_raw = [list(x) for x in test_df['words'].tolist()]
test_tags_raw  = [list(x) for x in test_df['labels'].tolist()]

test_tags = encode_tags(test_tags_raw)
test_obs   = encode_words_baseline(test_words_raw, baseline_vocab)

In [27]:
baseline_model = HMM(num_states=len(NER_TAGS), vocab_size=len(baseline_vocab))
baseline_model.train(train_tags, train_obs)
print("Baseline training complete.")


Baseline training complete.


In [28]:

baseline_preds = [baseline_model.predict(seq) for seq in test_obs]

baseline_acc = np.mean([np.array_equal(p, t) for p, t in zip(baseline_preds, test_tags)])
#flatten
true_flat = [t for seq in test_tags  for t in seq]
pred_flat= [t for seq in baseline_preds for t in seq]

print("\nBaseline Classification Report:")
print(classification_report(
    true_flat, pred_flat,
    labels=list(range(len(NER_TAGS))),
    target_names=NER_TAGS,
    zero_division=0
))



Baseline Classification Report:
                    precision    recall  f1-score   support

     B-ACCOUNTNAME       0.91      0.99      0.95       103
   B-ACCOUNTNUMBER       0.00      0.00      0.00       104
B-CREDITCARDNUMBER       0.00      0.00      0.00       105
           B-EMAIL       0.00      0.00      0.00       137
              B-IP       0.00      0.00      0.00        75
            B-IPV4       0.00      0.00      0.00       118
            B-IPV6       0.00      0.00      0.00       103
             B-MAC       0.00      0.00      0.00        79
        B-PASSWORD       0.00      0.00      0.00       109
    B-PHONE_NUMBER       1.00      0.01      0.02       117
             B-SSN       0.00      0.00      0.00        79
        B-USERNAME       0.00      0.00      0.00       139
     I-ACCOUNTNAME       0.77      0.99      0.87       184
   I-ACCOUNTNUMBER       0.00      0.00      0.00         0
I-CREDITCARDNUMBER       0.00      0.00      0.00         4
      

In [29]:
from seqeval.metrics import classification_report as seqeval_report
from seqeval.metrics import f1_score
print("\nSeqeval Classification Report:")
print(seqeval_report(
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in test_tags],
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in baseline_preds],
    zero_division=0
))
print("Span-Level F1:", f1_score(
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in test_tags],
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in baseline_preds],
    average='micro'
))


Seqeval Classification Report:
                  precision    recall  f1-score   support

     ACCOUNTNAME       0.79      0.84      0.82       106
   ACCOUNTNUMBER       0.00      0.00      0.00       104
CREDITCARDNUMBER       0.00      0.00      0.00       107
           EMAIL       0.00      0.00      0.00       137
              IP       0.00      0.00      0.00        75
            IPV4       0.00      0.00      0.00       118
            IPV6       0.00      0.00      0.00       103
             MAC       0.00      0.00      0.00        79
        PASSWORD       0.00      0.00      0.00       109
    PHONE_NUMBER       1.00      0.01      0.02       120
             SSN       0.00      0.00      0.00        79
        USERNAME       0.00      0.00      0.00       139

       micro avg       0.80      0.07      0.13      1276
       macro avg       0.15      0.07      0.07      1276
    weighted avg       0.16      0.07      0.07      1276

Span-Level F1: 0.12958963282937364


In [30]:
# Zhou & Su features 
def extract_word_features(word):
    feats = []

    for n in range(1, 5):
        if len(word) >= n:
            feats.append(f"PRE{n}_{word[:n].lower()}")
            feats.append(f"SUF{n}_{word[-n:].lower()}")

    if word.isupper():
        feats.append("ALL_CAPS")
    elif word[0].isupper():
        feats.append("INIT_CAP")
    elif any(c.isupper() for c in word):
        feats.append("HAS_CAP")
    else:
        feats.append("NO_CAP")

    if word.isdigit():
        feats.append("ALL_DIGITS")
    elif any(c.isdigit() for c in word):
        feats.append("HAS_DIGIT")

    if re.fullmatch(r'[\w.+-]+@[\w-]+\.[a-zA-Z]+', word):
        feats.append("IS_EMAIL")
    if re.fullmatch(r'\d{1,3}(\.\d{1,3}){3}', word):
        feats.append("IS_IPV4")
    if re.fullmatch(r'([0-9a-fA-F]{1,4}:){2,7}[0-9a-fA-F]{1,4}', word):
        feats.append("IS_IPV6")
    if re.fullmatch(r'([0-9a-fA-F]{2}[:\-]){5}[0-9a-fA-F]{2}', word):
        feats.append("IS_MAC")
    if re.fullmatch(r'\d{3}-\d{2}-\d{4}', word):
        feats.append("IS_SSN")
    if re.fullmatch(r'[\d\-]{13,19}', word):
        feats.append("IS_CREDITCARD")
    if re.fullmatch(r'\+?[\d\s\-\(\)]{7,15}', word):
        feats.append("IS_PHONE")
    if re.fullmatch(r'[A-Za-z0-9@#$%^&+=!]{6,}', word):
        feats.append("LOOKS_PASSWORD")

    l = len(word)
    feats.append("LEN_SHORT" if l <= 3 else "LEN_MED" if l <= 7 else "LEN_LONG")

    return feats


In [31]:
# build new vocabulary 
enhanced_vocab = {}

def register(tok):
    if tok not in enhanced_vocab:
        enhanced_vocab[tok] = len(enhanced_vocab)

for seq in train_words_raw:
    for w in seq:
        register(w)
        for f in extract_word_features(w):
            register(f)

enhanced_vocab['UNK'] = len(enhanced_vocab)


In [32]:
def word_to_feature_indices(word, vocab):
    tokens = [word] + extract_word_features(word)
    return [vocab.get(t, vocab['UNK']) for t in tokens]

def encode_words_enhanced(word_seqs, vocab):
    return [[word_to_feature_indices(w, vocab) for w in seq] for seq in word_seqs]


In [33]:

train_obs_enhanced = encode_words_enhanced(train_words_raw, enhanced_vocab)
test_obs_enhanced  = encode_words_enhanced(test_words_raw,  enhanced_vocab)

train_tags_enhanced = train_tags
test_tags_enhanced  = test_tags

print(f"Enhanced vocab size: {len(enhanced_vocab)}")


Enhanced vocab size: 138676


In [34]:
#train
enhanced_model = HMM(num_states=len(NER_TAGS), vocab_size=len(enhanced_vocab))
enhanced_model.train(train_tags_enhanced, train_obs_enhanced)
print("Enhanced training complete.")


Enhanced training complete.


In [35]:

enhanced_preds = [enhanced_model.predict(seq) for seq in test_obs_enhanced]

enhanced_acc = np.mean([np.array_equal(p, t) for p, t in zip(enhanced_preds, test_tags_enhanced)])
# flatten
true_flat_e = [t for seq in test_tags_enhanced    for t in seq]
pred_flat_e = [t for seq in enhanced_preds for t in seq]

print("\nEnhanced Classification Report:")
print(classification_report(
    true_flat_e, pred_flat_e,
    labels=list(range(len(NER_TAGS))),
    target_names=NER_TAGS,
    zero_division=0
))



Enhanced Classification Report:
                    precision    recall  f1-score   support

     B-ACCOUNTNAME       0.95      1.00      0.98       103
   B-ACCOUNTNUMBER       0.17      0.84      0.28       104
B-CREDITCARDNUMBER       0.28      0.66      0.40       105
           B-EMAIL       0.78      0.99      0.87       137
              B-IP       0.44      0.11      0.17        75
            B-IPV4       0.34      0.97      0.51       118
            B-IPV6       0.53      0.66      0.59       103
             B-MAC       0.70      0.73      0.72        79
        B-PASSWORD       0.31      0.72      0.43       109
    B-PHONE_NUMBER       0.40      0.90      0.55       117
             B-SSN       0.62      0.81      0.70        79
        B-USERNAME       0.42      0.40      0.41       139
     I-ACCOUNTNAME       0.94      0.99      0.97       184
   I-ACCOUNTNUMBER       0.00      0.00      0.00         0
I-CREDITCARDNUMBER       0.00      0.00      0.00         4
      

In [39]:
from seqeval.metrics import classification_report as seqeval_report
from seqeval.metrics import f1_score
print("\nSeqeval Classification Report:")
print(seqeval_report(
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in test_tags_enhanced],
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in enhanced_preds],
    zero_division=0
))
print("Span-Level F1:", f1_score(
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in test_tags_enhanced],
    [[INDEX_TO_NER_TAG[t] for t in seq] for seq in enhanced_preds],

    average='micro'
))


Seqeval Classification Report:
                  precision    recall  f1-score   support

     ACCOUNTNAME       0.88      0.94      0.91       106
   ACCOUNTNUMBER       0.17      0.84      0.28       104
CREDITCARDNUMBER       0.29      0.65      0.40       107
           EMAIL       0.78      0.99      0.87       137
              IP       0.44      0.11      0.17        75
            IPV4       0.34      0.97      0.51       118
            IPV6       0.53      0.66      0.59       103
             MAC       0.70      0.73      0.72        79
        PASSWORD       0.31      0.72      0.43       109
    PHONE_NUMBER       0.36      0.87      0.51       120
             SSN       0.53      0.70      0.60        79
        USERNAME       0.42      0.40      0.41       139

       micro avg       0.39      0.73      0.51      1276
       macro avg       0.48      0.72      0.53      1276
    weighted avg       0.48      0.73      0.54      1276

Span-Level F1: 0.5093979842004902
